In [3]:
##############################################################
### IMPORT ALL PACKAGES: MAKE SURE TO USE THE SE3nv KERNEL ###
##############################################################

# Standard library imports
import glob
import json
import math
import os
import random
import re
import shutil
import statistics
import string
import subprocess
import sys
import textwrap
import warnings
from collections import Counter, defaultdict, OrderedDict
import concurrent.futures 
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from itertools import permutations
import time
import multiprocessing

# Third-party imports
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import to_rgba
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import pyrosetta
import pyrosetta.distributed.tasks.rosetta_scripts as rosetta_scripts
from scipy.stats import gaussian_kde
from datetime import datetime
from Bio.PDB import PDBParser, PPBuilder
from Bio import pairwise2
from Bio import PDB
from Bio.SeqUtils import seq1
from Bio.pairwise2 import format_alignment
from IPython.display import display, HTML
from difflib import SequenceMatcher


In [4]:
#############################################
### ORGANIZATION & SETTING UP DIRECTORIES ###
#############################################

project_name = 'HAp_binder_design'

# Variable to control the display settings
display_all = True  # Set this to False if you want to revert to default display settings

# Define input paths and working directory
working_dir = '/home/tracyy2/HA_binder_design/07_01_2024_design2/' ### CHANGE
motifs_dir = '/home/tracyy2/HA_binder_design/HAp/output_ty/' ###change
os.chdir(working_dir)
os.makedirs(motifs_dir, exist_ok=True)

# Define output directory
output_dir = '/net/scratch/tracyy2/HAp_binder_design_outputs/07_01_2024_design2/' ### CHANGE

# Function to create directories and generate directory variables
def setup_directories(base_dir, dirs_list):
    for dir_name in dirs_list:
        full_path = os.path.join(base_dir, dir_name)
        os.makedirs(full_path, exist_ok=True)
        dir_var_name = dir_name.replace("/", "") + "_dir"
        globals()[dir_var_name] = full_path
            
# Set up working directories and associated variables
wrk_dirs_list = ['cmds/', 'submit/', 'logs/', 'cute_pics/', 'configs/', 'scores/',
                'fixed_residue_jsonls/', 'important_dfs/', 'looking/']
wrk_dirs_vars = setup_directories(working_dir, wrk_dirs_list)

# Set up output directories (scratch) and associated variables
out_dirs_list = ['diffusion_out/', 'partial_diffusion_out/', 'mpnn_out/', 'mpnn_with_ligand/', 'af2_out/',
                 'chemnet_cofactor_out/', 'chemnet_substrate_out/', 'chemnet_input_structures/',
                 'temporary_scores/', 'holo_af2_out/', 'relax_out/', 'chemnet_complex_out/', 
                 'inpainting_out/', 'predesign_out/']
out_dirs_vars = setup_directories(output_dir, out_dirs_list)

# Define the ligand name, input motifs, & params file
#params_file = '/home/woodbuse/eosin/in/EAY.params'

# Import Parameters and initialize PyRosetta
#pyrosetta.init(f"-mute all -in:file:extra_res_fa {params_file}")

# Set pandas display options based on the display_all variable
if display_all:
    pd.set_option('display.max_columns', None)  # Display all columns
    pd.set_option('display.max_rows', None)  # Display all rows
    pd.set_option('display.max_colwidth', None)  # Display full content of each cell
else:
    pd.reset_option('display.max_columns')
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_colwidth')
    
print(diffusion_out_dir)

/net/scratch/tracyy2/HAp_binder_design_outputs/07_01_2024_design2/diffusion_out/


In [5]:
###############################
### SET UP RF FLOW MATCHING ###
###############################

### SET OUTPUT DIRECTORY FOR GENERATED .PDB STRUCTURES & INPUT MOTIFS + LIG NAME ###
specific_diffusion_output_dir = f'{diffusion_out_dir}HAp_binder_design_240628_run/'
further_prefix = '_FM_model45' #want to add extra prefix before '.pdb' ? If not, write ''

input_motif_with_residues = glob.glob(f'{motifs_dir}HAp_010_OH_2rows_ty.pdb')
print('Number of input motifs =', len(input_motif_with_residues))

#Get an error of (`A`, `) not in the list, probably because res1 has no backbone
### SET UP CONTIGS FOR TIP ATOM RFAA FLOW MATCHING ###
contigs = ['30-75,A1-1,A2-2,A3-3,A4-4,A5-5,A7-7,A8-8,A9-9,A10-10,A11-11,A13-13,A14-14,A15-15,A16-16,A17-17'] # fragements to fix ['min_length-max_length,chainXresidueY-chainXresidueY+n,...']
contig_atoms = '{"A1":"OE2,OE1,CD","A2":"OE2,OE1,CD","A3":"OE2,OE1,CD","A4":"OE1,OE2,CD","A5":"OE2,OE1,CD","A7":"OE2,OE1,CD","A8":"OE2,OE1,CD","A9":"OE2,OE1,CD","A10":"OE2,OE1,CD","A11":"OE2,OE1,CD","A13":"OE2,OE1,CD","A14":"OE2,OE1,CD","A15":"OE2,OE1,CD","A16":"OE2,OE1,CD","A17":"OE2,OE1,CD"}' #fixed tip atoms
# contig_atoms = '{"A17":"OE2,OE1","A18":"OE2,OE1","A22":"OE2,OE1","A23":"OE2,OE1","A26":"OE2,OE1","A27":"OE2,OE1","A33":"OD2,OD1","A37":"OD2,OD1"}' #fixed tip atoms
length_range = "30-75" # min-max length of protein, should match first range of contigs

### ESSENTIAL RF FLOW MATCHING PARAMETERS ###
timesteps = 150 # basically the smaller this number is, the bigger the denoising steps taken
final_step = 1 # DECIDE IF 25-50, higher = less compute time but cuts number of timesteps

nstruct = 50 # total outputs you want per topology
nstruct_per_job = 2 # HOW MANY EACH COMMAND WILL RUN
num_trajectories = int(nstruct / nstruct_per_job)

### BONUS FEATURES TO TOGGLE IN FLOW MATCHING ###
guidepost_bonds = True
align_motif = False
str_self_cond = True
contig_as_guidepost = True
remove_guideposts_from_output = True
infer_guidepost_positions = True
guidepost_xyz_as_design = True
flexible_ligand = False
model_state_dict = 'model_state_dict' # or final_state_dict but might be thing of past
recenter_xt = True # new
recenter_xt_dropt = '3' # ONLY ONE NUMBER HERE, how many steps to recenter for
deterministic = False # MAKE THIS FALSE OR ALL OUTPUTS WILL BE SAME ONLY FOR BENCHMARKING
idealize_sidechain_outputs = True #DEFAULT IS FALSE SO MAKE THIS TRUE

### SET POTENTIALS? THIS MIGHT BE OUTDATED BUT CAN HELP WITH LIGAND CONTACT ###
include_potentials = False  # Set to True if you want to include the potentials section
potentials_list = ['ligand_ncontacts'] 
guide_decay = 'linear' #linear or quadratic, quadratic seems preferred these days
guide_scale = 2
weight = 0.65 #Previously 0.6

### SET SCRIPT & CHECKPOINT (SHOULDN'T CHANGE MUCH, CHECK FOR RECENT CHECKPOINT UPDATE) ###

checkpoint = os.getenv("RF_CHECKPOINT", "/net/databases/lab/aa-diffusion/models/RFD_45.pt")
script = os.getenv("RF_SCRIPT", "/home/tracyy2/git/rf_diffusion_repo_saman_recenter/rf_diffusion/run_inference.py")

#######################
### AUTOMATED STUFF ###
#######################

#lig_name = 'CA'

### GENERATE OUR CONFIGS & WRITE OUR COMMANDS ###
os.makedirs(specific_diffusion_output_dir, exist_ok=True)
commands_file = f'{cmds_dir}diffusion_{further_prefix}_{project_name}_cmds'
commands = commands_file
commands_bn = os.path.basename(commands[len(cmds_dir):])

#       ligand: {lig_name} INCLUDE BACK UNDER inference if you include a ligand

def generate_config_content(base_name, iteration, include_potentials):
    # Start with the common configuration content
    config_content = textwrap.dedent(f"""
    defaults:
      - base

    diffuser:
      T: {timesteps}
      
    guidepost_bonds: {guidepost_bonds}

    inference:
      align_motif: {align_motif}
      ckpt_path: {checkpoint}
      num_designs: {nstruct_per_job}
      output_prefix: {specific_diffusion_output_dir}{project_name}_{base_name}_{iteration}{further_prefix}
      input_pdb: {os.path.join(os.path.dirname(input_motif_with_residues[0]), f"{base_name}.pdb")}
      final_step: {final_step}
      contig_as_guidepost: {contig_as_guidepost}
      remove_guideposts_from_output: {remove_guideposts_from_output}
      infer_guidepost_positions: {infer_guidepost_positions}
      guidepost_xyz_as_design: {guidepost_xyz_as_design}
      flexible_ligand: {flexible_ligand}
      state_dict_to_load: {model_state_dict}
      recenter_xt: {recenter_xt}
      deterministic: {deterministic}
      recenter_xt_dropt: {recenter_xt_dropt}
      idealize_sidechain_outputs: {idealize_sidechain_outputs}

    contigmap:
      contigs: {contigs}
      length: '{length_range}'
      contig_atoms: '{contig_atoms}'
    """)

    # Add the potentials section only if potentials are defined
    if include_potentials == True:
        potentials_section = textwrap.dedent(f"""
        potentials:
          guiding_potentials: ["type:{potentials_list[0]},weight:{weight}"]
          guide_scale: {guide_scale}
          guide_decay: {guide_decay}
        """)
        config_content += potentials_section
    return config_content

# Make sure to use the correct variable (include_potentials) when calling generate_config_content
with open(commands_file, 'w') as f_out:
    for pdb in input_motif_with_residues:
        base_name = os.path.basename(pdb).replace('.pdb', '')
        for j in range(1, num_trajectories + 1):
            config_file = f'{configs_dir}{base_name}_{j}.yaml'
            with open(config_file, 'w') as config_f:
                config_txt = generate_config_content(base_name, j, include_potentials)
                config_f.write(config_txt)
            
            cmd = f'{script} --config-dir={configs_dir} --config-name={base_name}_{j}.yaml \n'
            f_out.write(cmd)
            
### SET UP THE JOB TO RUN ON DIGS ###
job_name = f"{commands_bn}_batch_job"
time = '04:00:00'
memory = '16g'
cores = '1'
queue = 'gpu'
cmds_per_job = 15

# should just work, no user input needed or wanted
num_jobs = int((((nstruct / nstruct_per_job) * len(input_motif_with_residues))) / cmds_per_job)+1
submit_file = f'{submit_dir}{job_name}.sh'
print("Number of Jobs =", num_jobs)
print('')
print("Commands File for Testing:")
print(commands)
print('')

# automated generation of submit script
rainier.submit_array_job(commands,time,cores,job_name,memory,submit_file,logs_dir,num_jobs,cmds_per_job,queue)

Number of input motifs = 1
Number of Jobs = 2

Commands File for Testing:
/home/tracyy2/HA_binder_design/07_01_2024_design2/cmds/diffusion__FM_model45_HAp_binder_design_cmds

submit this: 
/home/tracyy2/HA_binder_design/07_01_2024_design2/submit/diffusion__FM_model45_HAp_binder_design_cmds_batch_job.sh
